In [ ]:
# ==============================================================================
# CELL 1: CÀI ĐẶT MÔI TRƯỜNG & ĐỌC DỮ LIỆU
# ==============================================================================
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import mean_absolute_error, r2_score
import math

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Thiết bị: {device}")

TRAIN_PATH = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/train.csv'
VAL_PATH   = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/val.csv'
TEST_PATH  = '/kaggle/input/datasets/traanfddinhfkhair/datasetttt/data/test.csv'

df_train_raw = pd.read_csv(TRAIN_PATH)
df_val_raw   = pd.read_csv(VAL_PATH)
df_test_raw  = pd.read_csv(TEST_PATH)

df_final_all = pd.concat([df_train_raw, df_val_raw, df_test_raw], ignore_index=True)
df_final_all['TIME'] = pd.to_datetime(df_final_all['TIME'])
df_final_all = df_final_all.sort_values('TIME').reset_index(drop=True)

n_train = len(df_train_raw)
n_val   = len(df_val_raw)
n_test  = len(df_test_raw)

print(f"Dữ liệu gốc: Train: {n_train:,} | Val: {n_val:,} | Test: {n_test:,}")

In [ ]:
# ==============================================================================
# CELL 2: FEATURE ENGINEERING (FIXED LEAKAGE & TOPOLOGY FEATURES)
# ==============================================================================

# 1. Weather Forecast Shift (-48 steps = 24h)
weather_cols = ['temp', 'rhum', 'prcp', 'wspd']
forecast_weather_features = []
for col in weather_cols:
    new_col = f'{col}_forecast'
    df_final_all[new_col] = df_final_all[col].shift(-48)
    forecast_weather_features.append(new_col)

# 2. Cyclical Time Encoding
def encode_cyclical(df, col, max_val):
    df[col + '_sin'] = np.sin(2 * np.pi * df[col] / max_val)
    df[col + '_cos'] = np.cos(2 * np.pi * df[col] / max_val)
    return df

df_final_all['Hour']      = df_final_all['TIME'].dt.hour
df_final_all['DayOfWeek'] = df_final_all['TIME'].dt.dayofweek
df_final_all['Month']     = df_final_all['TIME'].dt.month
df_final_all = encode_cyclical(df_final_all, 'Hour', 24)
df_final_all = encode_cyclical(df_final_all, 'DayOfWeek', 7)
df_final_all = encode_cyclical(df_final_all, 'Month', 12)
df_final_all['is_weekend'] = (df_final_all['DayOfWeek'] >= 5).astype(float)
cyclical_cols = ['Hour_sin', 'Hour_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Month_sin', 'Month_cos', 'is_weekend']

# 3. Lag & Rolling Features
lag_features = []
for l in [2, 4, 6, 12, 48, 336]:
    col = f'P224_lag{l}'
    df_final_all[col] = df_final_all['P_224'].shift(l)
    lag_features.append(col)

df_final_all['P224_roll24']  = df_final_all['P_224'].shift(1).rolling(24).mean()
df_final_all['P224_roll48']  = df_final_all['P_224'].shift(1).rolling(48).mean()
df_final_all['P224_std24']   = df_final_all['P_224'].shift(1).rolling(24).std()
df_final_all['P224_std48']   = df_final_all['P_224'].shift(1).rolling(48).std()
df_final_all['P224_delta1']  = df_final_all['P_224'].diff(1).shift(1)
df_final_all['P224_delta48'] = df_final_all['P_224'].diff(48).shift(1)
lag_features += ['P224_roll24', 'P224_roll48', 'P224_std24', 'P224_std48', 'P224_delta1', 'P224_delta48']

# --- 🔥 FIX LEAKAGE: KHÔNG DÙNG BFILL ---
df_final_all = df_final_all.ffill() 
len_before = len(df_final_all)
df_final_all = df_final_all.dropna().reset_index(drop=True) 
rows_dropped = len_before - len(df_final_all)
n_train = n_train - rows_dropped # Quan trọng để Cell 3 không lệch

# 4. Định nghĩa nhóm Features
target_col = ['P_224']
substation_cols = [col for col in df_final_all.columns if col.startswith('P_') and col != 'P_224' and '_forecast' not in col]

# Extra Time Features
df_final_all['Hour_float'] = df_final_all['TIME'].dt.hour + df_final_all['TIME'].dt.minute / 60.0
df_final_all['DayOfMonth_sin'] = np.sin(2 * np.pi * df_final_all['TIME'].dt.day / 31)
df_final_all['DayOfMonth_cos'] = np.cos(2 * np.pi * df_final_all['TIME'].dt.day / 31)
df_final_all['WeekOfYear_sin'] = np.sin(2 * np.pi * df_final_all['TIME'].dt.isocalendar().week.astype(int) / 52)
df_final_all['WeekOfYear_cos'] = np.cos(2 * np.pi * df_final_all['TIME'].dt.isocalendar().week.astype(int) / 52)
extra_time_cols = ['Hour_float', 'DayOfMonth_sin', 'DayOfMonth_cos', 'WeekOfYear_sin', 'WeekOfYear_cos']

base_features = forecast_weather_features + cyclical_cols + extra_time_cols + lag_features
topo_features = base_features + substation_cols

print(f"Hoàn tất FE. Đã loại bỏ {rows_dropped} dòng mồ côi ở đầu.")

In [ ]:
# ==============================================================================
# CELL 3: CẮT DATA & FIT SCALER (CHỈ FIT TRÊN TRAIN)
# ==============================================================================
WINDOW_SIZE = 48 

df_train = df_final_all.iloc[:n_train].copy()
df_val   = df_final_all.iloc[n_train - WINDOW_SIZE : n_train + n_val].copy()
df_test  = df_final_all.iloc[n_train + n_val - WINDOW_SIZE : ].copy()

scaler_X_base = StandardScaler().fit(df_train[base_features])
scaler_X_topo = StandardScaler().fit(df_train[topo_features])
scaler_y      = RobustScaler().fit(df_train[target_col])

print(f"Data Splitting Done. Train set: {len(df_train)} rows.")

In [ ]:
# ==============================================================================
# CELL 4: KIẾN TRÚC MÔ HÌNH (ĐÃ ĐỒNG BỘ GELU, L1 & FULL BASELINES)
# ==============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# 1. Graph Encoder (Topo logic)
class SparseTopologyEncoder(nn.Module):
    def __init__(self, seq_len, num_features, topo_dim=32, num_layers=2, dropout=0.1, topk=4):
        super().__init__()
        self.num_features, self.topo_dim, self.topk = num_features, topo_dim, topk
        self.depthwise_conv = nn.Conv1d(num_features, num_features, kernel_size=3, padding=1, groups=num_features)
        self.temporal_proj = nn.Linear(seq_len, topo_dim)
        self.act, self.dropout = nn.GELU(), nn.Dropout(dropout)
        self.msg_mlps = nn.ModuleList([nn.Sequential(nn.Linear(topo_dim, topo_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(topo_dim, topo_dim)) for _ in range(num_layers)])
        self.norms = nn.ModuleList([nn.LayerNorm(topo_dim) for _ in range(num_layers)])
        self.gates = nn.ModuleList([nn.Sequential(nn.Linear(topo_dim, topo_dim), nn.Sigmoid()) for _ in range(num_layers)])
        self.pool_proj = nn.Linear(topo_dim * 2, topo_dim)
        self.node_emb = nn.Parameter(torch.randn(num_features, topo_dim) * 0.1)

    def build_adj(self, device):
        sim = torch.matmul(self.node_emb, self.node_emb.t()) / math.sqrt(self.topo_dim)
        A = torch.relu(sim) + torch.eye(self.num_features, device=device)
        if self.topk < self.num_features:
            vals, idx = torch.topk(A, k=max(1, self.topk), dim=-1)
            mask = torch.zeros_like(A).scatter_(1, idx, 1.0)
            A = A * mask
        return A / A.sum(dim=-1, keepdim=True).clamp_min(1e-6)

    def forward(self, x):
        h = self.act(self.temporal_proj(self.depthwise_conv(x.transpose(1, 2))))
        A_b = self.build_adj(x.device).unsqueeze(0).expand(h.size(0), -1, -1)
        global_ctx = h.mean(dim=1, keepdim=True)
        for msg_mlp, gate, norm in zip(self.msg_mlps, self.gates, self.norms):
            h = norm(h + self.dropout(self.act(msg_mlp(torch.bmm(A_b, h) + global_ctx) * gate(h))))
        return self.pool_proj(torch.cat([h.mean(dim=1), h.max(dim=1).values], dim=-1))

# 2. Lớp cha quản lý PHẠT L1 (Ngang hàng KAN)
class BaseTimeModel(nn.Module):
    def __init__(self, seq_len, num_features, output_dim):
        super().__init__()
        self.flatten_dim = seq_len * num_features
        self.ar_heads = nn.ModuleList([nn.Linear(self.flatten_dim, 1) for _ in range(output_dim)])
        
    def forward_ar(self, x):
        return torch.cat([head(x.reshape(x.size(0), -1)) for head in self.ar_heads], dim=1)

    def regularization_loss(self, lamb_l1=0.005):
        l1_reg = sum(torch.abs(p).mean() for name, p in self.named_parameters() if 'weight' in name)
        return lamb_l1 * l1_reg

# 3. Baselines Models Cũ (MLP, LSTM, CNN)
class MLP_Base(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64):
        super().__init__(seq_len, num_features, output_dim)
        self.net = nn.Sequential(nn.Linear(self.flatten_dim, hidden_dim*2), nn.GELU(), nn.Dropout(0.1),
                                 nn.Linear(hidden_dim*2, hidden_dim), nn.GELU(), nn.Linear(hidden_dim, output_dim))
    def forward(self, x): return self.forward_ar(x) + self.net(x.reshape(x.size(0), -1))

class LSTM_Base(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64):
        super().__init__(seq_len, num_features, output_dim)
        self.lstm = nn.LSTM(num_features, hidden_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x): 
        _, (h, _) = self.lstm(x)
        return self.forward_ar(x) + self.fc(h[-1])

class CNN_Base(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64):
        super().__init__(seq_len, num_features, output_dim)
        self.conv = nn.Sequential(nn.Conv1d(num_features, hidden_dim, 3, padding=1), nn.GELU(),
                                 nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1), nn.GELU(), nn.AdaptiveAvgPool1d(1))
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x): return self.forward_ar(x) + self.fc(self.conv(x.transpose(1, 2)).squeeze(-1))

# =========================================================
# 🔥 [NEW] BỔ SUNG MÔ HÌNH: GRU, TCN, iTransformer
# =========================================================

# 3.4. GRU_Base
class GRU_Base(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64):
        super().__init__(seq_len, num_features, output_dim)
        self.gru = nn.GRU(num_features, hidden_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x): 
        _, h = self.gru(x)
        return self.forward_ar(x) + self.fc(h[-1])

# 3.5. TCN_Base (Temporal Convolutional Network)
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TCN_Base(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64):
        super().__init__(seq_len, num_features, output_dim)
        # 2 blocks of dilated convolutions
        self.tcn = nn.Sequential(
            nn.Conv1d(num_features, hidden_dim, kernel_size=3, dilation=1, padding=2), Chomp1d(2), nn.GELU(), nn.Dropout(0.1),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, dilation=2, padding=4), Chomp1d(4), nn.GELU(), nn.Dropout(0.1)
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        h = self.tcn(x.transpose(1, 2)) # [B, num_features, seq_len] -> [B, hidden_dim, seq_len]
        return self.forward_ar(x) + self.fc(h[:, :, -1]) # Lấy output tại timestep cuối cùng

# 3.6. iTransformer_Base (Inverted Transformer)
# Cơ chế của iTransformer là coi mỗi "Biến" (Feature) là 1 token độc lập thay vì coi mỗi TimeStep là 1 token.
class iTransformer_Base(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64):
        super().__init__(seq_len, num_features, output_dim)
        self.project_time = nn.Linear(seq_len, hidden_dim) # Project d_time -> d_model
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=4, dim_feedforward=hidden_dim*2, 
                                                   dropout=0.1, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.flatten = nn.Flatten()
        self.fc_out = nn.Sequential(
            nn.Linear(num_features * hidden_dim, hidden_dim), 
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        # iTransformer input shape: [Batch, Num_Features, Seq_Len]
        x_inv = x.transpose(1, 2) 
        
        # Nhúng sequence length thành hidden_dim cho từng feature
        h = self.project_time(x_inv) # [B, Num_Features, hidden_dim]
        
        # Transformer tự học tương quan giữa CÁC BIẾN (Tương tự Topo)
        h = self.transformer(h) 
        
        # Flatten và dự báo
        h = self.flatten(h)
        return self.forward_ar(x) + self.fc_out(h)

# 4. MultiHead_Topo_MLP (Đối thủ của Topo-KAN)
class MultiHead_Topo_MLP(BaseTimeModel):
    def __init__(self, seq_len, num_features, output_dim, hidden_dim=64, topo_dim=32, fusion_dim=128, dropout=0.1):
        super().__init__(seq_len, num_features, output_dim)
        self.topo_encoder = SparseTopologyEncoder(seq_len, num_features, topo_dim=topo_dim, dropout=dropout)
        self.fusion = nn.Sequential(nn.Linear(seq_len * num_features + topo_dim, fusion_dim), nn.GELU(), nn.Dropout(dropout))
        self.layer1_mlp = nn.Sequential(nn.Linear(fusion_dim, hidden_dim), nn.GELU())
        self.layer2_mlp = nn.Linear(hidden_dim, output_dim)
        self.topo_to_hidden, self.topo_to_out_gate = nn.Linear(topo_dim, hidden_dim), nn.Sequential(nn.Linear(topo_dim, output_dim), nn.Sigmoid())

    def forward(self, x):
        topo_vec = self.topo_encoder(x)
        fused = self.fusion(torch.cat([x.reshape(x.size(0), -1), topo_vec], dim=1))
        hidden = self.layer1_mlp(fused) + self.topo_to_hidden(topo_vec)
        return self.forward_ar(x) + self.topo_to_out_gate(topo_vec) * self.layer2_mlp(hidden)

print("Đã nạp thành công toàn bộ mô hình: MLP, LSTM, CNN, GRU, TCN, iTransformer & MultiHead_Topo_MLP.")

In [ ]:
# ==============================================================================
# CELL 5: PIPELINE TRAIN & ĐÁNH GIÁ (FIXED CRASH & 30P CONTINUOUS)
# ==============================================================================
def asymmetric_loss(y_pred, y_true, penalty_factor=3.0):
    error = y_true - y_pred
    return torch.where(error > 0, penalty_factor * (error ** 2), (error ** 2)).mean()

def calculate_metrics(act, pre):
    mae = mean_absolute_error(act, pre)
    wape = np.sum(np.abs(act - pre)) / (np.sum(np.abs(act)) + 1e-10) * 100
    return mae, wape, r2_score(act, pre)

def train_dynamic_model(model_obj, name, loader, val_loader, device, epochs=100):
    optimizer = optim.Adam(model_obj.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    best_loss, counter, ckpt = float('inf'), 0, f"best_{name}.pth"

    for epoch in range(epochs):
        model_obj.train()
        train_losses = []
        for bx, by in loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            # 🔥 PHẠT NGANG HÀNG 0.005
            loss = asymmetric_loss(model_obj(bx), by, 3.0) + model_obj.regularization_loss(0.005)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_obj.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model_obj.eval()
        with torch.no_grad():
            val_l = sum(asymmetric_loss(model_obj(bx.to(device)), by.to(device), 3.0).item() for bx, by in val_loader) / len(val_loader)
        scheduler.step(val_l)
        
        if val_l < best_loss:
            best_loss, counter = val_l, 0
            torch.save(model_obj.state_dict(), ckpt)
        elif (counter := counter + 1) >= 15: break

    model_obj.load_state_dict(torch.load(ckpt, map_location=device))
    return model_obj

def run_dl_benchmarking(forecast_steps, model_class, feat_type='base', epochs=100):
    active_features = topo_features if feat_type == 'topo' else base_features
    active_scaler_X = scaler_X_topo if feat_type == 'topo' else scaler_X_base
    
    def create_custom_seq(df):
        X_raw, y_raw = active_scaler_X.transform(df[active_features]), scaler_y.transform(df[target_col])
        X, y, limit = [], [], len(df) - WINDOW_SIZE - forecast_steps + 1
        for i in range(limit):
            y_seq = y_raw[i + WINDOW_SIZE : i + WINDOW_SIZE + forecast_steps].flatten()
            if len(y_seq) == forecast_steps:
                X.append(X_raw[i : i + WINDOW_SIZE]); y.append(y_seq)
        return torch.from_numpy(np.array(X, dtype=np.float32)), torch.from_numpy(np.array(y, dtype=np.float32))

    train_loader = DataLoader(TensorDataset(*create_custom_seq(df_train)), batch_size=64, shuffle=True)
    val_loader   = DataLoader(TensorDataset(*create_custom_seq(df_val)),   batch_size=64, shuffle=False)
    test_loader  = DataLoader(TensorDataset(*create_custom_seq(df_test)),  batch_size=64, shuffle=False)
    
    model = model_class(WINDOW_SIZE, len(active_features), forecast_steps).to(device)
    model = train_dynamic_model(model, f"{model_class.__name__}_{feat_type}_{forecast_steps}P", train_loader, val_loader, device, epochs)
    
    model.eval()
    actuals, preds = [], []
    with torch.no_grad():
        for bx, by in test_loader:
            preds.append(model(bx.to(device)).cpu().numpy()); actuals.append(by.numpy())

    # 🔥 FIX CRASH SCALER
    inv_act = scaler_y.inverse_transform(np.concatenate(actuals).reshape(-1, 1)).reshape(-1, forecast_steps)
    inv_pre = scaler_y.inverse_transform(np.concatenate(preds).reshape(-1, 1)).reshape(-1, forecast_steps)
    
    # 🔥 NON-OVERLAP EVALUATION
    act_non_overlap = inv_act[::forecast_steps, :].flatten()
    pre_non_overlap = inv_pre[::forecast_steps, :].flatten()
    
    mae, wape, r2 = calculate_metrics(act_non_overlap, pre_non_overlap)
    print(f" RESULT {model_class.__name__} ({feat_type.upper()}) {forecast_steps}P: WAPE: {wape:.2f}% | MAE: {mae:.2f}")
    return {"Type": feat_type.upper(), "Model": model_class.__name__, "Points": forecast_steps, "Hours": forecast_steps*0.5, "MAE": mae, "WAPE (%)": wape, "R2": r2}

In [ ]:
# ==============================================================================
# CELL 6: VÒNG LẶP CHẠY SO SÁNH (FULL BASE + TOPO + TOPO-MLP)
# ==============================================================================
horizons = [ 48]
final_results = []

# 🔥 Bổ sung GRU, TCN, iTransformer vào danh sách đánh giá
dl_models = [ GRU_Base, TCN_Base, iTransformer_Base]

# BƯỚC 1 & 2: Đánh giá BASE và TOPO phẳng cho TẤT CẢ các Models
for p in horizons:
    for m_class in dl_models:
        for f_type in ['base', 'topo']:
            try:
                set_seed(42)
                final_results.append(run_dl_benchmarking(p, m_class, f_type))
            except Exception as e:
                print(f"Lỗi khi train {m_class.__name__} ({f_type.upper()}) tại mốc {p}P: {e}")
                continue

# BƯỚC 3: Chạy MultiHead_Topo_MLP (Đối thủ nặng ký nhất của KAN)
for p in horizons:
    try:
        set_seed(42)
        final_results.append(run_dl_benchmarking(p, MultiHead_Topo_MLP, 'topo'))
    except Exception as e:
        print(f"Lỗi khi train MultiHead_Topo_MLP tại mốc {p}P: {e}")
        continue

# IN BẢNG KẾT QUẢ TỔNG HỢP
if final_results:
    df_report = pd.DataFrame(final_results)
    df_report['WAPE_val'] = df_report['WAPE (%)'] 
    df_report['WAPE (%)'] = df_report['WAPE (%)'].apply(lambda x: f"{x:.2f}%")
    df_report = df_report.sort_values(by=['Points', 'WAPE_val']).drop(columns=['WAPE_val'])
    
    print("\n" + "="*80 + "\n BẢNG SO SÁNH ĐẦY ĐỦ CÁC MÔ HÌNH BASELINES (NGANG HÀNG KAN) \n" + "="*80)
    print(df_report.to_string(index=False))